# 02.0 Baseline ImageNet models

## Notebook overview
This notebook runs the two ImageNet-based feature baselines, PatchCore and PaDiM, using the shared split manifest created in notebook 01.

## Purpose
- establish strong pretrained feature baselines before introducing self-supervised learning
- reuse the exact train, validation, and test split manifest from notebook 01 rather than rebuilding the split inside this notebook
- evaluate ImageNet PatchCore and ImageNet PaDiM under the same image size, backbone, and reporting framework
- save clean category-level and mean-level baseline tables for later comparison notebooks
- export a compact summary figure and a small qualitative heatmap figure for GitHub and report use

## Process
1. resolve the dataset path and load the split manifest produced by notebook 01.
2. build the shared image transforms, dataset classes, dataloaders, and evaluation helpers.
3. fit and evaluate ImageNet PatchCore and ImageNet PaDiM for each active category.
4. save baseline tables, settings, and figures for later notebooks in the repo.

## Notes
- run notebook 01 first so the split manifest and leakage report already exist
- this notebook does not recreate the split, which keeps the baseline stage cleaner and less error-prone
- the saved tables are intended to be reused later by the autoencoder notebook and the main comparison notebook


In [ ]:
#------------------------------------------------------------------------------
# 1.0 Optional dependency install
#------------------------------------------------------------------------------
# Install FAISS only if it is not already available in the runtime.
try:
    import faiss
    print("faiss is already available.")
except Exception:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
    import faiss
    print("Installed faiss-cpu.")


In [ ]:
#------------------------------------------------------------------------------
# 1.1 Imports and notebook helpers
#------------------------------------------------------------------------------
import os
import sys
import json
import time
import math
import random
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from sklearn.metrics import roc_auc_score, average_precision_score

from IPython.display import display


# Print a clean section banner so the notebook output is easier to follow.
def print_banner(text):
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)


# Create a parent folder before saving an artefact.
def ensure_parent(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


# Save JSON with overwrite behaviour.
def save_json_overwrite(obj, path):
    path = ensure_parent(path)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


# Save a DataFrame to CSV with overwrite behaviour.
def save_csv_overwrite(df, path):
    path = ensure_parent(path)
    df.to_csv(path, index=False)


# Read JSON from disk.
def read_json(path):
    with open(Path(path), "r") as f:
        return json.load(f)


# Return a file size in MB.
def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2) if path.exists() else np.nan


# Set Python, NumPy, and PyTorch seeds.
def set_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# Build a stable integer seed from a string key.
def stable_seed_from_text(seed, text):
    key = f"{seed}_{text}".encode("utf-8")
    return int(hashlib.md5(key).hexdigest()[:8], 16)


# Reset the tracked peak CUDA memory before a stage.
def reset_peak_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


# Read the peak CUDA memory in MB.
def get_peak_gpu_memory_mb():
    return float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else np.nan


print_banner("Environment")
print("Python        :", sys.version.split()[0])
print("Torch         :", torch.__version__)
print("Torchvision   :", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count     :", torch.cuda.device_count())
for gpu_idx in range(torch.cuda.device_count()):
    print(f"GPU {gpu_idx} name   :", torch.cuda.get_device_name(gpu_idx))


# 2.0 Run settings

## Purpose
- keep the baseline configuration explicit near the top of the notebook
- make the notebook portable across Kaggle and local execution
- store all key output paths before training begins
- keep the main comparison settings matched to the final report settings


In [ ]:
#------------------------------------------------------------------------------
# 2.1 Core configuration and output paths
#------------------------------------------------------------------------------
NOTEBOOK_ID = "02_baseline_imagenet_models"
RUN_MODE = "full"                  # "debug" or "full"
SEED = 42
DETERMINISTIC_DEBUG = False
set_seed(SEED, deterministic=(RUN_MODE == "debug" and DETERMINISTIC_DEBUG))

DEBUG_CATEGORIES = ["bottle", "carpet", "tile", "transistor"]
CATEGORIES_ALL = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper",
]
CATEGORIES_ACTIVE = DEBUG_CATEGORIES if RUN_MODE == "debug" else CATEGORIES_ALL
QUAL_CATEGORY = next((cat for cat in ["carpet", "tile", "bottle", "transistor"] if cat in CATEGORIES_ACTIVE), CATEGORIES_ACTIVE[0])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_COUNT = torch.cuda.device_count()

IMG_SIZE = 224
VAL_FRAC = 0.10
BATCH_SIZE_TRAIN = 32
BATCH_SIZE_TEST = 16

BACKBONE_NAME = "resnet18_imagenet"
PATCH_LAYER_NAMES = ["layer3"]
PADIM_LAYER_NAME = "layer3"
CORESET_RATIO = 0.05
MAX_TRAIN_PATCHES = 120_000
PATCH_SCORE_TOPK = 200
PADIM_DIM = 100
PADIM_EPS = 1e-4

CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS_BASE = min(4, max(2, CPU_COUNT // 2))
NUM_WORKERS = NUM_WORKERS_BASE if DEVICE == "cuda" else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working/industrial_anomaly_detection_mvtec")
else:
    WORK_ROOT = Path.cwd() / "industrial_anomaly_detection_mvtec"

NOTEBOOK_ROOT = WORK_ROOT / NOTEBOOK_ID / RUN_MODE
FIGURES_DIR = NOTEBOOK_ROOT / "figures"
TABLES_DIR = NOTEBOOK_ROOT / "tables"
JSON_DIR = NOTEBOOK_ROOT / "json"

for folder in [FIGURES_DIR, TABLES_DIR, JSON_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PRIOR_NOTEBOOK_ROOT = WORK_ROOT / "01_dataset_audit_and_splits" / RUN_MODE
SPLIT_MANIFEST_PATH = PRIOR_NOTEBOOK_ROOT / "json" / "split_manifest.json"
LEAKAGE_REPORT_PATH = PRIOR_NOTEBOOK_ROOT / "json" / "leakage_report.json"

RUN_METADATA_PATH = JSON_DIR / "run_metadata.json"
BASELINE_SETTINGS_PATH = JSON_DIR / "baseline_settings.json"

TABLE_BASELINES_IMAGENET_CATEGORY_PATH = TABLES_DIR / "table_baselines_imagenet_category.csv"
TABLE_BASELINES_IMAGENET_MEAN_PATH = TABLES_DIR / "table_baselines_imagenet_mean.csv"
TABLE_BASELINES_IMAGENET_FULL_PATH = TABLES_DIR / "table_baselines_imagenet.csv"

FIG_BASELINES_IMAGENET_PATH = FIGURES_DIR / "fig_baselines_imagenet.png"
FIG_HEATMAPS_IMAGENET_PATH = FIGURES_DIR / "fig_heatmaps_imagenet.png"

run_metadata = {
    "notebook_id": NOTEBOOK_ID,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": DEVICE,
    "gpu_count": GPU_COUNT,
    "categories_active": CATEGORIES_ACTIVE,
    "img_size": IMG_SIZE,
    "batch_size_train": BATCH_SIZE_TRAIN,
    "batch_size_test": BATCH_SIZE_TEST,
    "num_workers": NUM_WORKERS,
    "work_root": str(WORK_ROOT),
    "prior_notebook_root": str(PRIOR_NOTEBOOK_ROOT),
}
save_json_overwrite(run_metadata, RUN_METADATA_PATH)

print_banner("Run configuration")
print("RUN_MODE          :", RUN_MODE)
print("DEVICE            :", DEVICE)
print("GPU_COUNT         :", GPU_COUNT)
print("CATEGORIES_ACTIVE :", CATEGORIES_ACTIVE)
print("WORK_ROOT         :", WORK_ROOT)
print("NOTEBOOK_ROOT     :", NOTEBOOK_ROOT)


# 3.0 Dataset and prior artefacts

## Purpose
- resolve the MVTec AD dataset path in a portable way
- load the split manifest from notebook 01 instead of recreating it here
- verify that notebook 01 was run successfully and that its leakage checks were clean


In [ ]:
#------------------------------------------------------------------------------
# 3.1 Resolve the dataset path and load notebook 01 artefacts
#------------------------------------------------------------------------------
DATASET_CANDIDATES = [
    Path(os.environ.get("MVTEC_DIR", "")) if os.environ.get("MVTEC_DIR") else None,
    Path("/kaggle/input/datasets/ipythonx/mvtec-ad"),
    Path("/kaggle/input/mvtec-ad"),
    Path.cwd() / "data" / "raw" / "mvtec-ad",
]

MVTEC_DIR = None
for candidate in DATASET_CANDIDATES:
    if candidate is not None and candidate.exists():
        MVTEC_DIR = candidate
        break

if MVTEC_DIR is None:
    raise FileNotFoundError(
        "MVTec AD dataset path was not found. Set MVTEC_DIR as an environment variable "
        "or attach the dataset in Kaggle before running the notebook."
    )

if not SPLIT_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Split manifest was not found at {SPLIT_MANIFEST_PATH}. "
        "Run 01_dataset_audit_and_splits.ipynb first."
    )

if not LEAKAGE_REPORT_PATH.exists():
    raise FileNotFoundError(
        f"Leakage report was not found at {LEAKAGE_REPORT_PATH}. "
        "Run 01_dataset_audit_and_splits.ipynb first."
    )

splits = read_json(SPLIT_MANIFEST_PATH)
leakage_report = read_json(LEAKAGE_REPORT_PATH)

if not leakage_report.get("all_checks_zero", False):
    raise AssertionError(
        "Notebook 01 reported non-zero leakage checks. Fix that before running the baseline notebook."
    )

missing_categories = sorted(set(CATEGORIES_ACTIVE) - set(splits.keys()))
if len(missing_categories) > 0:
    raise ValueError(
        f"The split manifest does not contain the required categories for this run: {missing_categories}"
    )

available_categories = sorted([path.name for path in MVTEC_DIR.iterdir() if path.is_dir()])

print_banner("Loaded inputs")
print("MVTEC_DIR           :", MVTEC_DIR)
print("Available categories:", available_categories)
print("Split manifest path :", SPLIT_MANIFEST_PATH)
print("Leakage report path :", LEAKAGE_REPORT_PATH)
print("Leakage all zero    :", leakage_report.get("all_checks_zero"))

split_summary_rows = []
for category in CATEGORIES_ACTIVE:
    split_summary_rows.append(
        {
            "category": category,
            "train_good_n": len(splits[category]["train_good"]),
            "val_good_n": len(splits[category]["val_good"]),
            "test_total_n": len(splits[category]["test"]),
            "test_good_n": int(sum(int(row["label"]) == 0 for row in splits[category]["test"])),
            "test_anomaly_n": int(sum(int(row["label"]) == 1 for row in splits[category]["test"])),
        }
    )

df_split_summary = pd.DataFrame(split_summary_rows).sort_values("category").reset_index(drop=True)
display(df_split_summary)


# 4.0 Shared datasets, transforms, and evaluation helpers

## Purpose
- define the shared image transforms, dataset class, dataloaders, and evaluation helpers for the baseline stage
- keep model-specific cells shorter by moving repeated data handling and metric logic into one place
- make the notebook easier to audit and reuse later


In [ ]:
#------------------------------------------------------------------------------
# 4.1 Shared transforms, dataset class, dataloaders, and metrics
#------------------------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

tfm_imagenet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


# Load one RGB image from disk.
def load_rgb(path):
    return Image.open(path).convert("RGB")


# Load one binary mask from disk, or return an all-zero mask for normal images.
def load_mask(path):
    if path is None or (isinstance(path, float) and np.isnan(path)):
        return torch.zeros((1, IMG_SIZE, IMG_SIZE), dtype=torch.float32)

    mask = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST)
    mask = (np.array(mask, dtype=np.float32) > 0).astype(np.float32)
    return torch.from_numpy(mask)[None, :, :]


# Return items in a consistent format across train, validation, and test splits.
class MvtecDataset(Dataset):
    def __init__(self, items, mode, img_tfm):
        self.items = items
        self.mode = mode
        self.img_tfm = img_tfm

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        if self.mode in ["train_good", "val_good"]:
            img_path = item
            label = 0
            mask_path = None
        else:
            img_path = item["img_path"]
            label = int(item["label"])
            mask_path = item.get("mask_path", None)

        image = self.img_tfm(load_rgb(img_path))
        mask = load_mask(mask_path)
        return image, int(label), mask, str(img_path)


# Build a DataLoader using settings matched to the active runtime.
def make_loader(items, mode, img_tfm, batch_size, shuffle):
    dataset = MvtecDataset(items=items, mode=mode, img_tfm=img_tfm)

    loader_kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE == "cuda",
    }

    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

    return DataLoader(**loader_kwargs)


# Build the train, validation, and test loaders for one category.
def make_split_loaders(category):
    train_loader = make_loader(
        items=splits[category]["train_good"],
        mode="train_good",
        img_tfm=tfm_imagenet,
        batch_size=BATCH_SIZE_TRAIN,
        shuffle=True,
    )
    val_loader = make_loader(
        items=splits[category]["val_good"],
        mode="val_good",
        img_tfm=tfm_imagenet,
        batch_size=BATCH_SIZE_TEST,
        shuffle=False,
    )
    test_loader = make_loader(
        items=splits[category]["test"],
        mode="test",
        img_tfm=tfm_imagenet,
        batch_size=BATCH_SIZE_TEST,
        shuffle=False,
    )
    return train_loader, val_loader, test_loader


# Compute image-level ROC-AUC safely.
def safe_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Compute image-level PR-AUC safely.
def safe_pr_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else average_precision_score(y_true, y_score)


# Flatten masks and heatmaps to compute pixel ROC-AUC.
def pixel_roc_auc(masks_list, heatmaps_list):
    if len(masks_list) == 0 or len(heatmaps_list) == 0:
        return np.nan

    y_true = np.concatenate([mask.reshape(-1) for mask in masks_list]).astype(int)
    y_score = np.concatenate([heat.reshape(-1) for heat in heatmaps_list]).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Rescale an array to the 0 to 1 range for plotting.
def norm_01(x):
    x = x.astype(np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)


# Convert a normalised tensor image back to a display image.
def tensor_to_display(img_tensor):
    image = img_tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = image * np.array(IMAGENET_STD)[None, None, :] + np.array(IMAGENET_MEAN)[None, None, :]
    return np.clip(image, 0.0, 1.0)


# Draw an image with an optional heatmap overlay.
def overlay(ax, img_tensor, heat_2d=None, title=""):
    ax.imshow(tensor_to_display(img_tensor))
    if heat_2d is not None:
        ax.imshow(norm_01(heat_2d), alpha=0.45)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


# Draw a grayscale mask.
def show_mask(ax, mask_2d, title=""):
    ax.imshow(mask_2d, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


# 5.0 Baseline model helpers

## Purpose
- define the shared ResNet backbone, feature hooks, PatchCore logic, PaDiM logic, and evaluation function
- keep the actual experiment loop focused on per-category execution and result collection
- match the main design choices used in the final report


In [ ]:
#------------------------------------------------------------------------------
# 5.1 Backbone, feature hooks, PatchCore, PaDiM, and evaluation helpers
#------------------------------------------------------------------------------
def get_resnet_imagenet():
    model = torchvision.models.resnet18(
        weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1
    )
    model.fc = nn.Identity()
    model.eval()
    return model


# Register hooks so intermediate feature maps can be collected.
def make_feature_hooks(model, layer_names):
    features = {}
    handles = []

    def hook_factory(layer_name):
        def _hook(_, __, output):
            features[layer_name] = output
        return _hook

    for layer_name in layer_names:
        handle = getattr(model, layer_name).register_forward_hook(hook_factory(layer_name))
        handles.append(handle)

    return features, handles


# Remove hook handles after the experiment finishes.
def remove_handles(handles):
    for handle in handles:
        handle.remove()


@torch.inference_mode()
def forward_get_feats(model, features, inputs, layer_names):
    _ = model(inputs)
    return [features[layer_name] for layer_name in layer_names]


# Flatten a feature map into patch rows.
def fmap_to_patches(feature_map):
    batch_n, channels, height, width = feature_map.shape
    return feature_map.permute(0, 2, 3, 1).reshape(batch_n, height * width, channels)


# Upsample and concatenate patch features from one or more layers.
def concat_patch_features(feature_maps):
    target_h, target_w = feature_maps[-1].shape[-2:]
    patch_list = []

    for feature_map in feature_maps:
        if feature_map.shape[-2:] != (target_h, target_w):
            feature_map = F.interpolate(
                feature_map,
                size=(target_h, target_w),
                mode="bilinear",
                align_corners=False,
            )
        patch_list.append(fmap_to_patches(feature_map))

    return torch.cat(patch_list, dim=-1)


# Build a FAISS L2 index for nearest-neighbour search.
def faiss_index_l2(array_2d):
    array_2d = np.asarray(array_2d, dtype=np.float32)
    index = faiss.IndexFlatL2(array_2d.shape[1])
    index.add(array_2d)
    return index


# Collect normal training patch embeddings and keep a deterministic coreset subset.
def build_patch_bank(model, features, train_loader, layer_names, category):
    patch_chunks = []
    total_patches = 0

    for images, _, _, _ in train_loader:
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        feature_maps = forward_get_feats(model, features, images, layer_names)
        patches = concat_patch_features(feature_maps).detach().cpu().numpy()
        patches = patches.reshape(-1, patches.shape[-1]).astype(np.float32)
        patch_chunks.append(patches)
        total_patches += patches.shape[0]

        if total_patches >= MAX_TRAIN_PATCHES:
            break

    bank_full = np.concatenate(patch_chunks, axis=0).astype(np.float32)

    if len(bank_full) > MAX_TRAIN_PATCHES:
        rng_cap = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_patch_bank_cap"))
        idx_cap = rng_cap.choice(len(bank_full), size=MAX_TRAIN_PATCHES, replace=False)
        bank_full = bank_full[idx_cap]

    keep_n = max(1, int(round(len(bank_full) * float(CORESET_RATIO))))
    keep_n = min(keep_n, len(bank_full))

    rng_core = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_patchcore_coreset"))
    coreset_idx = rng_core.choice(len(bank_full), size=keep_n, replace=False)
    bank = bank_full[coreset_idx]

    return bank


@torch.inference_mode()
def patchcore_scores(model, features, images, layer_names, index):
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
    feature_maps = forward_get_feats(model, features, images, layer_names)
    patches = concat_patch_features(feature_maps).detach().cpu().numpy()

    batch_n, patch_n, feat_dim = patches.shape
    queries = patches.reshape(-1, feat_dim).astype(np.float32)
    dist2, _ = index.search(queries, 1)
    dist2 = dist2.reshape(batch_n, patch_n)

    feat_h, feat_w = feature_maps[-1].shape[-2:]
    heat = dist2.reshape(batch_n, feat_h, feat_w)
    heat_tensor = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(
        heat_tensor,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1).numpy()

    flat = heat_up.reshape(batch_n, -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heat_up


# Fit PaDiM Gaussian statistics from normal training patches.
def padim_fit(model, features, train_loader, layer_name, category):
    all_feature_maps = []

    for images, _, _, _ in train_loader:
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        feature_map = forward_get_feats(model, features, images, [layer_name])[0].detach().cpu()
        all_feature_maps.append(feature_map)

    feature_map = torch.cat(all_feature_maps, dim=0)
    sample_n, channels, feat_h, feat_w = feature_map.shape

    dim = min(PADIM_DIM, channels)
    rng_dim = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_padim_dim"))
    dim_idx = rng_dim.choice(channels, size=dim, replace=False)

    feature_map = feature_map[:, dim_idx, :, :]
    feats_np = feature_map.permute(0, 2, 3, 1).reshape(sample_n, feat_h * feat_w, dim).numpy()

    mu = feats_np.mean(axis=0)
    cov_inv = []

    eye = np.eye(dim, dtype=np.float32)
    for patch_idx in range(feat_h * feat_w):
        patch_features = feats_np[:, patch_idx, :]
        cov = np.cov(patch_features, rowvar=False).astype(np.float32) + PADIM_EPS * eye
        cov_inv.append(np.linalg.inv(cov).astype(np.float32))

    cov_inv = np.stack(cov_inv, axis=0)

    return {
        "dim_idx": dim_idx.astype(np.int64),
        "mu": mu.astype(np.float32),
        "cov_inv": cov_inv.astype(np.float32),
        "H": int(feat_h),
        "W": int(feat_w),
        "D": int(dim),
    }


@torch.inference_mode()
def padim_scores(model, features, images, layer_name, stats):
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
    feature_map = forward_get_feats(model, features, images, [layer_name])[0].detach().cpu()

    dim_idx = torch.tensor(stats["dim_idx"])
    feature_map = feature_map[:, dim_idx, :, :]

    batch_n, feat_dim, feat_h, feat_w = feature_map.shape
    feats_np = feature_map.permute(0, 2, 3, 1).reshape(batch_n, feat_h * feat_w, feat_dim).numpy()

    mu = stats["mu"]
    cov_inv = stats["cov_inv"]

    heat = np.zeros((batch_n, feat_h * feat_w), dtype=np.float32)
    for patch_idx in range(feat_h * feat_w):
        diff = feats_np[:, patch_idx, :] - mu[patch_idx]
        tmp = diff @ cov_inv[patch_idx]
        heat[:, patch_idx] = np.sum(tmp * diff, axis=1)

    heat = heat.reshape(batch_n, feat_h, feat_w)
    heat_tensor = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(
        heat_tensor,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1).numpy()

    flat = heat_up.reshape(batch_n, -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heat_up


# Evaluate one scoring function across the test set.
def eval_method(test_loader, score_fn):
    y_true = []
    y_score = []
    masks = []
    heats = []

    start_time = time.time()
    eval_image_n = 0

    for images, labels, batch_masks, _ in test_loader:
        scores, heatmaps = score_fn(images)

        y_true.extend(labels.numpy().tolist())
        y_score.extend(scores.tolist())

        mask_np = batch_masks.squeeze(1).numpy()
        for idx in range(mask_np.shape[0]):
            masks.append(mask_np[idx])
            heats.append(heatmaps[idx])

        eval_image_n += images.shape[0]

    total_sec = time.time() - start_time

    return {
        "img_roc_auc": safe_roc_auc(y_true, y_score),
        "img_pr_auc": safe_pr_auc(y_true, y_score),
        "px_roc_auc": pixel_roc_auc(masks, heats),
        "sec_per_img": total_sec / max(eval_image_n, 1),
        "n_eval_images": int(eval_image_n),
        "y_true": y_true,
        "y_score": y_score,
        "masks": masks,
        "heats": heats,
    }


# Pick a small good versus anomaly subset for the qualitative figure.
def select_qual_items(test_items, n_good=3, n_bad=3):
    good_items = [item for item in test_items if int(item["label"]) == 0][:n_good]
    bad_items = [item for item in test_items if int(item["label"]) == 1][:n_bad]
    return good_items + bad_items


# 6.0 Run ImageNet PatchCore and PaDiM

## Purpose
- run the two ImageNet baselines category by category
- collect category-level metrics, mean metrics, and light resource statistics
- cache one small qualitative example set for the later figure


In [ ]:
#------------------------------------------------------------------------------
# 6.1 Run the ImageNet baseline models
#------------------------------------------------------------------------------
baseline_settings = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": DEVICE,
    "mvtec_dir": str(MVTEC_DIR),
    "split_manifest_path": str(SPLIT_MANIFEST_PATH),
    "categories_active": CATEGORIES_ACTIVE,
    "image_size": IMG_SIZE,
    "val_frac": VAL_FRAC,
    "batch_size_train": BATCH_SIZE_TRAIN,
    "batch_size_test": BATCH_SIZE_TEST,
    "backbone_name": BACKBONE_NAME,
    "patch_layer_names": PATCH_LAYER_NAMES,
    "padim_layer_name": PADIM_LAYER_NAME,
    "patchcore": {
        "coreset_ratio": CORESET_RATIO,
        "max_train_patches": MAX_TRAIN_PATCHES,
        "score_topk_pixels": PATCH_SCORE_TOPK,
    },
    "padim": {
        "dim": PADIM_DIM,
        "eps": PADIM_EPS,
        "score_topk_pixels": PATCH_SCORE_TOPK,
    },
    "stage": "imagenet_baselines_only",
}
save_json_overwrite(baseline_settings, BASELINE_SETTINGS_PATH)

print_banner("Build ImageNet backbone")
backbone = get_resnet_imagenet().to(DEVICE)
feature_names = sorted(set(PATCH_LAYER_NAMES + [PADIM_LAYER_NAME]))
features, handles = make_feature_hooks(backbone, feature_names)

baseline_rows = []
qual_cache = None

for category in CATEGORIES_ACTIVE:
    print("\n" + "-" * 90)
    print("CATEGORY:", category)
    print("-" * 90)

    train_items = splits[category]["train_good"]
    val_items = splits[category]["val_good"]
    test_items = splits[category]["test"]

    n_test_total = len(test_items)
    n_test_anomaly = int(sum(int(item["label"]) == 1 for item in test_items))

    train_loader, val_loader, test_loader = make_split_loaders(category)

    #----------------------------------------------------------------------
    # ImageNet PatchCore
    #----------------------------------------------------------------------
    reset_peak_gpu_memory()
    fit_start = time.time()

    patch_bank = build_patch_bank(
        model=backbone,
        features=features,
        train_loader=train_loader,
        layer_names=PATCH_LAYER_NAMES,
        category=category,
    )
    patch_index = faiss_index_l2(patch_bank)

    patchcore_fit_sec = time.time() - fit_start

    def score_fn_patchcore(images):
        return patchcore_scores(
            model=backbone,
            features=features,
            images=images,
            layer_names=PATCH_LAYER_NAMES,
            index=patch_index,
        )

    patchcore_eval = eval_method(test_loader, score_fn_patchcore)
    patchcore_peak_gpu_mem_mb = get_peak_gpu_memory_mb()

    baseline_rows.append(
        {
            "category": category,
            "method": "imagenet_patchcore",
            "representation_source": "imagenet",
            "anomaly_head": "patchcore",
            "aug_strength": "none",
            "backbone_name": BACKBONE_NAME,
            "layers": ",".join(PATCH_LAYER_NAMES),
            "img_size": IMG_SIZE,
            "coreset_ratio": CORESET_RATIO,
            "n_train_good": len(train_items),
            "n_val_good": len(val_items),
            "n_test_total": n_test_total,
            "n_test_anomaly": n_test_anomaly,
            "feature_dim": int(patch_bank.shape[1]),
            "memory_bank_n": int(patch_bank.shape[0]),
            "memory_bank_mb": float(patch_bank.nbytes / (1024 ** 2)),
            "fit_sec": patchcore_fit_sec,
            "sec_per_img": patchcore_eval["sec_per_img"],
            "n_eval_images": patchcore_eval["n_eval_images"],
            "img_roc_auc": patchcore_eval["img_roc_auc"],
            "img_pr_auc": patchcore_eval["img_pr_auc"],
            "px_roc_auc": patchcore_eval["px_roc_auc"],
            "peak_gpu_mem_mb": patchcore_peak_gpu_mem_mb,
            "checkpoint_size_mb": np.nan,
        }
    )

    print(
        "PatchCore:",
        {
            "img_roc_auc": patchcore_eval["img_roc_auc"],
            "img_pr_auc": patchcore_eval["img_pr_auc"],
            "px_roc_auc": patchcore_eval["px_roc_auc"],
            "fit_sec": patchcore_fit_sec,
            "sec_per_img": patchcore_eval["sec_per_img"],
        },
    )

    #----------------------------------------------------------------------
    # ImageNet PaDiM
    #----------------------------------------------------------------------
    reset_peak_gpu_memory()
    fit_start = time.time()

    padim_stats = padim_fit(
        model=backbone,
        features=features,
        train_loader=train_loader,
        layer_name=PADIM_LAYER_NAME,
        category=category,
    )
    padim_fit_sec = time.time() - fit_start

    padim_memory_mb = (padim_stats["mu"].nbytes + padim_stats["cov_inv"].nbytes) / (1024 ** 2)

    def score_fn_padim(images):
        return padim_scores(
            model=backbone,
            features=features,
            images=images,
            layer_name=PADIM_LAYER_NAME,
            stats=padim_stats,
        )

    padim_eval = eval_method(test_loader, score_fn_padim)
    padim_peak_gpu_mem_mb = get_peak_gpu_memory_mb()

    baseline_rows.append(
        {
            "category": category,
            "method": "imagenet_padim",
            "representation_source": "imagenet",
            "anomaly_head": "padim",
            "aug_strength": "none",
            "backbone_name": BACKBONE_NAME,
            "layers": PADIM_LAYER_NAME,
            "img_size": IMG_SIZE,
            "coreset_ratio": np.nan,
            "n_train_good": len(train_items),
            "n_val_good": len(val_items),
            "n_test_total": n_test_total,
            "n_test_anomaly": n_test_anomaly,
            "feature_dim": int(padim_stats["D"]),
            "memory_bank_n": int(padim_stats["H"] * padim_stats["W"]),
            "memory_bank_mb": float(padim_memory_mb),
            "fit_sec": padim_fit_sec,
            "sec_per_img": padim_eval["sec_per_img"],
            "n_eval_images": padim_eval["n_eval_images"],
            "img_roc_auc": padim_eval["img_roc_auc"],
            "img_pr_auc": padim_eval["img_pr_auc"],
            "px_roc_auc": padim_eval["px_roc_auc"],
            "peak_gpu_mem_mb": padim_peak_gpu_mem_mb,
            "checkpoint_size_mb": np.nan,
        }
    )

    print(
        "PaDiM:",
        {
            "img_roc_auc": padim_eval["img_roc_auc"],
            "img_pr_auc": padim_eval["img_pr_auc"],
            "px_roc_auc": padim_eval["px_roc_auc"],
            "fit_sec": padim_fit_sec,
            "sec_per_img": padim_eval["sec_per_img"],
        },
    )

    #----------------------------------------------------------------------
    # Cache qualitative examples for one chosen category
    #----------------------------------------------------------------------
    if category == QUAL_CATEGORY:
        qual_items = select_qual_items(test_items, n_good=3, n_bad=3)

        if len(qual_items) > 0:
            qual_loader = make_loader(
                items=qual_items,
                mode="test",
                img_tfm=tfm_imagenet,
                batch_size=len(qual_items),
                shuffle=False,
            )

            qual_images, qual_labels, qual_masks, qual_paths = next(iter(qual_loader))
            _, qual_heat_patchcore = patchcore_scores(
                model=backbone,
                features=features,
                images=qual_images,
                layer_names=PATCH_LAYER_NAMES,
                index=patch_index,
            )
            _, qual_heat_padim = padim_scores(
                model=backbone,
                features=features,
                images=qual_images,
                layer_name=PADIM_LAYER_NAME,
                stats=padim_stats,
            )

            qual_cache = {
                "category": category,
                "images": qual_images.cpu(),
                "labels": qual_labels.numpy(),
                "masks": qual_masks.squeeze(1).numpy(),
                "paths": list(qual_paths),
                "heat_patchcore": qual_heat_patchcore,
                "heat_padim": qual_heat_padim,
            }

    del patch_bank, patch_index, padim_stats
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

remove_handles(handles)

df_baselines_imagenet_category = pd.DataFrame(baseline_rows)

mean_cols = [
    "img_roc_auc",
    "img_pr_auc",
    "px_roc_auc",
    "fit_sec",
    "sec_per_img",
    "n_eval_images",
    "feature_dim",
    "memory_bank_n",
    "memory_bank_mb",
    "checkpoint_size_mb",
    "peak_gpu_mem_mb",
]

df_baselines_imagenet_mean = (
    df_baselines_imagenet_category
    .groupby(["method", "representation_source", "anomaly_head", "aug_strength"], as_index=False)[mean_cols]
    .mean(numeric_only=True)
)

df_baselines_imagenet_mean["category"] = "MEAN"
df_baselines_imagenet_mean["backbone_name"] = BACKBONE_NAME
df_baselines_imagenet_mean["layers"] = df_baselines_imagenet_mean["anomaly_head"].map(
    {"patchcore": ",".join(PATCH_LAYER_NAMES), "padim": PADIM_LAYER_NAME}
)
df_baselines_imagenet_mean["img_size"] = IMG_SIZE
df_baselines_imagenet_mean["coreset_ratio"] = df_baselines_imagenet_mean["anomaly_head"].map(
    {"patchcore": CORESET_RATIO, "padim": np.nan}
)
df_baselines_imagenet_mean["n_train_good"] = np.nan
df_baselines_imagenet_mean["n_val_good"] = np.nan
df_baselines_imagenet_mean["n_test_total"] = np.nan
df_baselines_imagenet_mean["n_test_anomaly"] = np.nan

ordered_cols = [
    "category",
    "method",
    "representation_source",
    "anomaly_head",
    "aug_strength",
    "backbone_name",
    "layers",
    "img_size",
    "coreset_ratio",
    "n_train_good",
    "n_val_good",
    "n_test_total",
    "n_test_anomaly",
    "img_roc_auc",
    "img_pr_auc",
    "px_roc_auc",
    "fit_sec",
    "sec_per_img",
    "n_eval_images",
    "feature_dim",
    "memory_bank_n",
    "memory_bank_mb",
    "checkpoint_size_mb",
    "peak_gpu_mem_mb",
]

df_baselines_imagenet_category = df_baselines_imagenet_category[ordered_cols].sort_values(
    ["method", "category"]
).reset_index(drop=True)
df_baselines_imagenet_mean = df_baselines_imagenet_mean[ordered_cols].sort_values(
    ["method", "category"]
).reset_index(drop=True)
df_baselines_imagenet_full = pd.concat(
    [df_baselines_imagenet_category, df_baselines_imagenet_mean],
    ignore_index=True,
)

save_csv_overwrite(df_baselines_imagenet_category, TABLE_BASELINES_IMAGENET_CATEGORY_PATH)
save_csv_overwrite(df_baselines_imagenet_mean, TABLE_BASELINES_IMAGENET_MEAN_PATH)
save_csv_overwrite(df_baselines_imagenet_full, TABLE_BASELINES_IMAGENET_FULL_PATH)

print_banner("ImageNet baseline tables")
display(df_baselines_imagenet_full.tail(10))
print("Saved category table :", TABLE_BASELINES_IMAGENET_CATEGORY_PATH)
print("Saved mean table     :", TABLE_BASELINES_IMAGENET_MEAN_PATH)
print("Saved full table     :", TABLE_BASELINES_IMAGENET_FULL_PATH)


# 7.0 Save baseline figures

## Purpose
- export one compact multi-metric summary figure for the two ImageNet baselines
- export one qualitative heatmap figure for the chosen qualitative category
- keep the outputs lightweight enough for GitHub while still showing what the methods are doing


In [ ]:
#------------------------------------------------------------------------------
# 7.1 Baseline summary figure and qualitative heatmaps
#------------------------------------------------------------------------------
method_order = ["imagenet_patchcore", "imagenet_padim"]

mean_plot_df = (
    df_baselines_imagenet_category
    .groupby("method")[["img_roc_auc", "img_pr_auc", "px_roc_auc", "sec_per_img"]]
    .mean(numeric_only=True)
    .reindex(method_order)
    .reset_index()
)

fig = plt.figure(figsize=(12, 8))

ax1 = plt.subplot(2, 2, 1)
ax1.bar(mean_plot_df["method"], mean_plot_df["img_roc_auc"])
ax1.set_title("Mean image ROC-AUC")
ax1.set_ylim(0, 1.02)
ax1.tick_params(axis="x", rotation=20)

ax2 = plt.subplot(2, 2, 2)
ax2.bar(mean_plot_df["method"], mean_plot_df["img_pr_auc"])
ax2.set_title("Mean image PR-AUC")
ax2.set_ylim(0, 1.02)
ax2.tick_params(axis="x", rotation=20)

ax3 = plt.subplot(2, 2, 3)
ax3.bar(mean_plot_df["method"], mean_plot_df["px_roc_auc"])
ax3.set_title("Mean pixel ROC-AUC")
ax3.set_ylim(0, 1.02)
ax3.tick_params(axis="x", rotation=20)

ax4 = plt.subplot(2, 2, 4)
ax4.bar(mean_plot_df["method"], mean_plot_df["sec_per_img"])
ax4.set_title("Mean inference sec / image")
ax4.tick_params(axis="x", rotation=20)

plt.tight_layout()
ensure_parent(FIG_BASELINES_IMAGENET_PATH)
plt.savefig(FIG_BASELINES_IMAGENET_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_BASELINES_IMAGENET_PATH)

if qual_cache is not None:
    qual_images = qual_cache["images"]
    qual_labels = qual_cache["labels"]
    qual_masks = qual_cache["masks"]
    qual_paths = qual_cache["paths"]
    qual_heat_patchcore = qual_cache["heat_patchcore"]
    qual_heat_padim = qual_cache["heat_padim"]

    row_n = qual_images.shape[0]
    plt.figure(figsize=(12, 3.0 * row_n))

    for row_idx in range(row_n):
        row_name = f"{'anomaly' if qual_labels[row_idx] == 1 else 'good'} | {Path(qual_paths[row_idx]).parent.name}"

        ax = plt.subplot(row_n, 4, row_idx * 4 + 1)
        overlay(ax, qual_images[row_idx], None, title="Image")
        ax.set_ylabel(row_name, fontsize=9)

        ax = plt.subplot(row_n, 4, row_idx * 4 + 2)
        show_mask(ax, qual_masks[row_idx], title="GT mask")

        ax = plt.subplot(row_n, 4, row_idx * 4 + 3)
        overlay(ax, qual_images[row_idx], qual_heat_patchcore[row_idx], title="PatchCore")

        ax = plt.subplot(row_n, 4, row_idx * 4 + 4)
        overlay(ax, qual_images[row_idx], qual_heat_padim[row_idx], title="PaDiM")

    plt.tight_layout()
    ensure_parent(FIG_HEATMAPS_IMAGENET_PATH)
    plt.savefig(FIG_HEATMAPS_IMAGENET_PATH, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close()

    print("Saved figure:", FIG_HEATMAPS_IMAGENET_PATH)
else:
    print("No qualitative cache was created, so the heatmap figure was skipped.")


# 8.0 Final review

## Purpose
- print the saved output paths in one place
- make it easy to confirm that the notebook finished with the expected artefacts


In [ ]:
#------------------------------------------------------------------------------
# 8.1 Final saved artefacts
#------------------------------------------------------------------------------
print_banner("Saved artefacts")
print("Run metadata path          :", RUN_METADATA_PATH)
print("Baseline settings path     :", BASELINE_SETTINGS_PATH)
print("Category table path        :", TABLE_BASELINES_IMAGENET_CATEGORY_PATH)
print("Mean table path            :", TABLE_BASELINES_IMAGENET_MEAN_PATH)
print("Full table path            :", TABLE_BASELINES_IMAGENET_FULL_PATH)
print("Summary figure path        :", FIG_BASELINES_IMAGENET_PATH)
print("Qualitative figure path    :", FIG_HEATMAPS_IMAGENET_PATH)
print("Split manifest dependency  :", SPLIT_MANIFEST_PATH)
print("Leakage report dependency  :", LEAKAGE_REPORT_PATH)
